# **Kaggle Challenge 2: Đề tài dự báo giá nhà.**
# **THỰC HIỆN DỰ ĐOÁN GIÁ NHÀ**
**Mục tiêu:** Dự đoán giá nhà trên tập test và xuất ra kết quả đúng định dạng để đăng lên Kaggle. Sử dụng mô hình tốt nhất để dự đoán trên tập test của Kaggle.

## 1. Import các thư viện cần thiết

In [27]:
import numpy as np
import pickle
import pandas as pd
import joblib

from sklearn.preprocessing import RobustScaler

## 2. Tải dữ liệu từ tập test đã xử lý

In [28]:
test_data = pd.DataFrame(pd.read_pickle("../preprocessing/data/test_processed.pkl"))

In [29]:
test_data.head()

,MSSubClass,LotFrontage,LotArea,LotShape,LandContour,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,...,ScreenPorchBinary,PoolAreaBinary,MiscValBinary,GarageAreaBinary,GarageCarsBinary,FireplacesBinary,FullBathBinary,HalfBathBinary,BsmtFullBathBinary,BsmtHalfBathBinary
0,20,0.464671,0.475099,3,3,2,5,6,0.644928,0.183333,...,1,0,0,1,1,0,1,0,0,0
1,20,0.507940,0.952946,2,3,2,6,6,0.623188,0.133333,...,0,0,1,1,1,0,1,1,0,0
2,60,0.193276,0.880449,2,3,2,5,5,0.905797,0.800000,...,0,0,0,1,1,1,1,1,0,0
3,60,0.376507,0.119678,2,3,2,6,6,0.913043,0.800000,...,0,0,0,1,1,1,1,1,0,0
4,120,-1.687348,-1.488062,2,1,2,8,5,0.869565,0.700000,...,1,0,0,1,1,0,1,0,0,0


## 3. Tải mô hình tốt nhất đã được huấn luyện

In [30]:
best_model = joblib.load("../models/saved_model.pkl") 

## 4. Thực hiện việc tái thiết lập các biến đổi của biến mục tiêu
Biến mục tiêu đã bị biến đổi trong quá trình tiền xử lý dữ liệu nhằm để cải thiện hiệu suất mô hình, do đó kết quả mô hình dự đoán cũng sẽ bị thay đổi. Cần đảo ngược các biến đổi này trước khi xuất ra file submission.

In [31]:
y_data_unprocessed = pd.DataFrame(pd.read_pickle("../preprocessing/data/train_y_unprocessed.pkl")) # Tải dữ liệu chứa biến mục tiêu chưa xử lý

In [32]:
y_data_unprocessed.head() # Kiểm tra

,SalePrice
0,208500
1,181500
2,223500
3,140000
4,250000


In [33]:
# Tái tiền xử lý dữ liệu để thiết lập lại scaler, log transformation.
scaler = RobustScaler()
y_data_unprocessed = np.log1p(y_data_unprocessed)
y_data_unprocessed = scaler.fit_transform(y_data_unprocessed.values.reshape(-1, 1))

## 5. Thực hiện dự đoán giá nhà bằng mô hình tốt nhất

In [34]:
y_test_pred = best_model.predict(test_data) # Dự đoán giá nhà trên test.

## 6. Thực hiện nghịch đảo lại các biến đổi của giá trị đã dự đoán
Biến đổi dữ liệu đã được biến đổi về lại dữ liệu ban đầu

In [35]:
# Nghịch đảo lại scaler, log transformation (do biến này đã được biến đổi trong tiền xử lý dữ liệu), trả về dữ liệu thật
y_pred_real = scaler.inverse_transform(y_test_pred.reshape(-1, 1)) # Đảo ngược RobustScaler trước (do RobustScaler làm sau log transform lúc chuẩn hóa)
y_pred_real = np.expm1(y_pred_real) # sau đó đảo ngược log transformation

# Chuyển thành vector 1 chiều
y_pred_real = y_pred_real.ravel() 

## 7. Lưu ra kết quả 

In [36]:
# Đọc file csv từ file sample
sample = pd.read_csv("../data/sample_submission.csv")

# Lưu file submission đúng dạng Kaggle
submission = pd.DataFrame({
    "Id": sample["Id"],
    "SalePrice": y_pred_real
})

submission.to_csv("submission.csv", index=False) # Xuất file submission
print("Đã tạo ra file submission.csv")

Đã tạo ra file submission.csv
